In [ ]:
# ==============================================================================
# ECONOMETRICS LAB: CAUSALITY TESTS & COMPARATIVE DYNAMICS (VIX II vs EPU)
# ==============================================================================

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

warnings.filterwarnings('ignore')

plt.style.use(
    'seaborn-v0_8-whitegrid'
    if 'seaborn-v0_8-whitegrid' in plt.style.available
    else 'default'
)


def format_spines(ax):
  ax.spines['top'].set_visible(False)
  ax.spines['bottom'].set_color('black')
  ax.spines['left'].set_color('black')
  ax.spines['right'].set_color('black')


print('=' * 80)
print(
    '--- [ECONOMETRICS BENCH]: CAUSALITY TESTS (VIX TUPINIQUIM II vs EPU'
    ' BRAZIL) ---'
)
print('=' * 80)

# ==============================================================================
# 1. HISTORICAL DATA INGESTION AND ALIGNMENT
# ==============================================================================
df_epu = pd.read_excel(
    'epu_baker_bloom_davis_brazil.xlsx', sheet_name='Brazil EPU Index'
)
df_epu['Date'] = pd.to_datetime(
    df_epu['year'].astype(str) + '-' + df_epu['month'].astype(str) + '-01'
)
df_epu = df_epu.rename(columns={'Brazil News-Based EPU': 'EPU'})

df_vix = pd.read_excel('vix_tupiniquim_ii_historical_series.xlsx')
df_vix['Date'] = pd.to_datetime(df_vix['Date'])

# ==============================================================================
# 2. STL STRUCTURAL FILTERING & DATA MERGE
# ==============================================================================
print('\nApplying STL Structural Filter (period=13) to Brazil EPU Index...')
stl_epu = STL(df_epu['EPU'], period=13, robust=True).fit()
df_epu['EPU_SA'] = stl_epu.trend + stl_epu.resid

df_analysis = (
    pd.merge(df_vix, df_epu[['Date', 'EPU_SA']], on='Date', how='inner')
    .sort_values('Date')
    .reset_index(drop=True)
)
print(
    f'[ALIGNMENT]: Common synchronized sample with {len(df_analysis)} monthly'
    ' observations.'
)

# ==============================================================================
# 3. STATIONARITY VERIFICATION (ADF TEST)
# ==============================================================================
print('\n' + '=' * 80)
print('                  1. AUGMENTED DICKEY-FULLER (ADF) TESTS')
print('=' * 80)

p_vix = adfuller(df_analysis['VIX_Tupiniquim_XGBoost'])[1]
p_epu = adfuller(df_analysis['EPU_SA'])[1]

print(
    '-> VIX Tupiniquim II (Level)        | ADF p-value: '
    f'{p_vix:.4f} -> Stationary I(0)'
)
print(
    '-> Deseasonalized EPU (Level)       | ADF p-value: '
    f'{p_epu:.4f} -> Stationary I(0)'
)
print('=' * 80)

# ==============================================================================
# 4. GRANGER CAUSALITY TEST IN LEVELS (BOTH SERIES ARE I(0))
# ==============================================================================
max_lags = 3
print('\n' + '=' * 80)
print(
    f'        2. GRANGER CAUSALITY IN LEVELS (Lags 1 to {max_lags})'
)
print('=' * 80)

# Direction 1: VIX Tupiniquim -> EPU_SA (Level)
print('\n[DIRECTION 1]: VIX Tupiniquim (Market) -> EPU (News/Media)')
gc_vix_to_epu = grangercausalitytests(
    df_analysis[['EPU_SA', 'VIX_Tupiniquim_XGBoost']],
    maxlag=max_lags,
    verbose=False,
)
for lag in range(1, max_lags + 1):
  p_val = gc_vix_to_epu[lag][0]['ssr_ftest'][1]
  decision = (
      'REJECTS H0 (Granger-Causes)'
      if p_val < 0.05
      else 'Fails to Reject H0 (No Causality)'
  )
  print(f'-> Lag {lag}: p-value = {p_val:.4f} | {decision}')

# Direction 2: EPU_SA (Level) -> VIX Tupiniquim
print('\n[DIRECTION 2]: EPU (News/Media) -> VIX Tupiniquim (Market)')
gc_epu_to_vix = grangercausalitytests(
    df_analysis[['VIX_Tupiniquim_XGBoost', 'EPU_SA']],
    maxlag=max_lags,
    verbose=False,
)
for lag in range(1, max_lags + 1):
  p_val = gc_epu_to_vix[lag][0]['ssr_ftest'][1]
  decision = (
      'REJECTS H0 (Granger-Causes)'
      if p_val < 0.05
      else 'Fails to Reject H0 (No Causality)'
  )
  print(f'-> Lag {lag}: p-value = {p_val:.4f} | {decision}')
print('=' * 80)

# ==============================================================================
# 5. TODA-YAMAMOTO (1995) PROCEDURE (ROBUSTNESS CHECK)
# ==============================================================================
print('\n' + '=' * 80)
print('            3. TODA-YAMAMOTO MODIFIED WALD TEST')
print('=' * 80)

d_max = 1
var_model = VAR(df_analysis[['VIX_Tupiniquim_XGBoost', 'EPU_SA']])
lag_order = var_model.select_order(maxlags=6)
k = max(lag_order.bic, 1)

print(f'-> Optimal VAR lag order (k): {k} lag(s)')
print(f'-> Maximum integration order (d_max): {d_max}')
print(f'-> Augmented VAR estimated with (k + d_max) = {k + d_max} lags in levels.\n')


def run_toda_yamamoto(df_data, y_name, x_name, k_lags, d_integration):
  df_ty = pd.DataFrame(index=df_data.index)
  df_ty['const'] = 1.0

  for i in range(1, k_lags + d_integration + 1):
    df_ty[f'{y_name}_lag{i}'] = df_data[y_name].shift(i)

  for i in range(1, k_lags + d_integration + 1):
    df_ty[f'{x_name}_lag{i}'] = df_data[x_name].shift(i)

  df_ty['target'] = df_data[y_name]
  df_reg = df_ty.dropna()

  X_mat = df_reg.drop(columns=['target'])
  y_vec = df_reg['target']

  ols_model = sm.OLS(y_vec, X_mat).fit(cov_type='HC1')

  restrictions = [f'{x_name}_lag{i} = 0' for i in range(1, k_lags + 1)]
  wald_formula = ', '.join(restrictions)

  wald_test = ols_model.wald_test(wald_formula, scalar=True)
  return wald_test.statistic, wald_test.pvalue


stat_1, p_val_1 = run_toda_yamamoto(
    df_analysis, 'EPU_SA', 'VIX_Tupiniquim_XGBoost', k, d_max
)
decision_1 = (
    'REJECTS H0 (Causes)'
    if p_val_1 < 0.05
    else 'Fails to Reject H0 (No Causality)'
)
print('[DIRECTION 1 (TY)]: VIX Tupiniquim -> EPU_SA (Level)')
print(
    f'-> Wald Statistic: {stat_1:.4f} | p-value = {p_val_1:.4f} | {decision_1}'
)

stat_2, p_val_2 = run_toda_yamamoto(
    df_analysis, 'VIX_Tupiniquim_XGBoost', 'EPU_SA', k, d_max
)
decision_2 = (
    'REJECTS H0 (Causes)'
    if p_val_2 < 0.05
    else 'Fails to Reject H0 (No Causality)'
)
print('\n[DIRECTION 2 (TY)]: EPU_SA (Level) -> VIX Tupiniquim')
print(
    f'-> Wald Statistic: {stat_2:.4f} | p-value = {p_val_2:.4f} | {decision_2}'
)
print('=' * 80)

# ==============================================================================
# 6. DUAL-AXIS COMPARISON CHART: VIX TUPINIQUIM II vs. EPU BRAZIL
# ==============================================================================
print(
    '\nGenerating Dual-Axis Comparison Chart: VIX Tupiniquim II vs. EPU'
    ' Brazil...'
)

fig, ax1 = plt.subplots(figsize=(14, 5.2))

# Axis 1: VIX Tupiniquim II (Dark Orange)
ax1.plot(
    df_analysis['Date'],
    df_analysis['VIX_Tupiniquim_XGBoost'],
    color='darkorange',
    linewidth=2.2,
)
ax1.set_xlabel('Years', fontweight='bold')
ax1.set_ylabel(
    'VIX Tupiniquim II (Points)', fontweight='bold', color='darkorange'
)
ax1.tick_params(axis='y', labelcolor='darkorange')
ax1.grid(True, linestyle=':', alpha=0.4)

# Axis 2: Seasonally Adjusted EPU (Blue Dashed)
ax2 = ax1.twinx()
ax2.plot(
    df_analysis['Date'],
    df_analysis['EPU_SA'],
    color='#1f77b4',
    linewidth=1.8,
    linestyle='--',
)
ax2.set_ylabel('EPU Index (Points)', fontweight='bold', color='#1f77b4')
ax2.tick_params(axis='y', labelcolor='#1f77b4')

format_spines(ax1)
format_spines(ax2)

plt.tight_layout()
plt.savefig('vix_vs_epu_en.png', dpi=300, bbox_inches='tight')
plt.show()

print("[SUCCESS]: 'vix_vs_epu_en.png' generated successfully!")